# Neobank Customer Churn Analysis
**Portfolio Project 2 — Ebinum**

---

## Project Overview

Customer churn — when a user closes their account or stops engaging — is one of the most costly problems a neobank faces. Acquiring a new customer is significantly more expensive than retaining an existing one.

This project analyses a synthetic dataset of **1,500 neobank customers** to answer three core business questions:

1. **Who is churning?** — Which customer segments have the highest churn rates?
2. **Why are they churning?** — Which behavioural signals most strongly predict churn?
3. **What should we do about it?** — Which customers should we prioritise for retention, and what is the revenue at stake?

### Dataset Features

| Feature | Description |
|---|---|
| `customer_id` | Unique customer identifier |
| `signup_date` | Date account was opened |
| `tenure_months` | How long the customer has been with the bank |
| `age` | Customer age |
| `region` | Canadian province |
| `plan_tier` | Free / Standard / Premium |
| `monthly_fee_cad` | Monthly subscription fee in CAD |
| `num_products` | Number of products held (chequing, savings, crypto, investments) |
| `avg_monthly_balance_cad` | Average account balance |
| `balance_trend` | Whether balance is growing, stable, or declining |
| `monthly_txn_count` | Average monthly transactions (last 3 months) |
| `app_logins_monthly` | Average monthly app logins |
| `days_since_last_login` | Days elapsed since most recent login |
| `support_contacts_6mo` | Number of support contacts in the last 6 months |
| `has_external_bank_link` | Whether customer has linked an external bank account |
| `referral_source` | How the customer was acquired |
| `churned` | Target variable — 1 = churned, 0 = retained |


---
## Section 1 — Environment Setup & Data Loading

### What this section does
Imports the necessary libraries and loads the dataset into a Pandas DataFrame. We also run an immediate data quality check to confirm there are no missing values before any analysis begins.

### Why this matters for interviews
Starting every analysis with a validation step shows analytical discipline. Analysts who skip this step often build conclusions on top of dirty data — a common failure point hiring managers look for.

### Libraries used
- **`pandas`** — the core library for data manipulation and analysis in Python
- **`numpy`** — used for numerical operations; also used during dataset generation
- **`warnings`** — suppresses non-critical Pandas display warnings that clutter notebook output


In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv("neobank_churn_dataset.csv")

# ── Data quality check ──
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"\nData types:")
print(df.dtypes)


Rows: 1,500 | Columns: 17
Missing values: 0
Duplicate rows: 0

Data types:
customer_id                    str
signup_date                    str
tenure_months                int64
age                          int64
region                         str
plan_tier                      str
monthly_fee_cad            float64
num_products                 int64
avg_monthly_balance_cad    float64
balance_trend                  str
monthly_txn_count            int64
app_logins_monthly           int64
days_since_last_login        int64
support_contacts_6mo         int64
has_external_bank_link       int64
referral_source                str
churned                      int64
dtype: object


### Reading the output
- **Rows / Columns** — confirms the full dataset loaded correctly (1,500 rows, 17 columns)
- **Missing values: 0** — no nulls; no imputation needed before analysis
- **Duplicate rows: 0** — no repeated records that would skew counts
- **Data types** — verifies numeric columns are numeric and string columns are object; wrong dtypes here (e.g. a fee column loaded as string) would break downstream calculations


---
## Section 2 — Dataset Overview

### What this section does
Uses `df.describe()` to generate a statistical summary of every column — counts, means, min/max for numeric columns, and unique value counts for categoricals. This is the standard first look at a new dataset.

### Why `include="all"` and `.T`
- `include="all"` extends describe beyond just numeric columns to include string/categorical columns (showing `count`, `unique`, `top`, `freq`)
- `.T` transposes the result — when you have many columns, transposing makes the summary vertical and far easier to read in a notebook
- Selecting `[["count","unique","mean","min","max"]]` trims noise from the output while keeping the most useful summary statistics


In [2]:
# Statistical summary — numeric and categorical columns together
summary = df.describe(include="all").T[["count", "unique", "mean", "min", "max"]]
summary["mean"] = summary["mean"].round(2)
print(summary.to_string())


                          count unique     mean   min       max
customer_id                1500   1500      NaN   NaN       NaN
signup_date                1500     71      NaN   NaN       NaN
tenure_months            1500.0    NaN    26.77   1.0      71.0
age                      1500.0    NaN    30.93  18.0      53.0
region                     1500      5      NaN   NaN       NaN
plan_tier                  1500      3      NaN   NaN       NaN
monthly_fee_cad          1500.0    NaN     8.26   0.0     19.99
num_products             1500.0    NaN     1.99   1.0       4.0
avg_monthly_balance_cad  1500.0    NaN  2016.44  0.79  29398.74
balance_trend              1500      3      NaN   NaN       NaN
monthly_txn_count        1500.0    NaN    16.98   2.0      46.0
app_logins_monthly       1500.0    NaN    11.34   0.0      34.0
days_since_last_login    1500.0    NaN    14.05   0.0     120.0
support_contacts_6mo     1500.0    NaN      1.2   0.0       7.0
has_external_bank_link   1500.0    NaN  

### Key observations from the summary
- `tenure_months` ranges from 1–71 — good spread across early, mid, and long-tenure customers
- `avg_monthly_balance_cad` has a wide range (near zero to ~$29k) — right-skewed distribution typical of balance data
- `monthly_fee_cad` shows 0 as the minimum, confirming free-tier customers are included
- `churned` has a mean of ~0.28, meaning approximately **28% of customers churned** — the overall churn rate we will reference throughout


---
## Section 3 — Churn Rate Overview

### What this section does
Establishes the baseline churn KPIs: total customers, churned count, retained count, and overall churn rate. This is always Step 1 in any churn analysis — you need the overall rate before any segmentation is meaningful.

### Why establish a baseline first
Segment-level churn rates (e.g. "Free-tier churns at 40%") are only meaningful when you can compare them against the overall rate (28%). Without a baseline, you cannot distinguish a high-risk segment from an average one.


In [3]:
total = len(df)
churned = df["churned"].sum()
retained = total - churned
churn_rate = churned / total

print("=" * 45)
print("CHURN BASELINE KPIs")
print("=" * 45)
print(f"  Total customers  : {total:,}")
print(f"  Churned          : {churned:,}")
print(f"  Retained         : {retained:,}")
print(f"  Overall churn rate: {churn_rate:.1%}")
print("=" * 45)


CHURN BASELINE KPIs
  Total customers  : 1,500
  Churned          : 418
  Retained         : 1,082
  Overall churn rate: 27.9%


### Reading the output
- ~**27.9% overall churn rate** is the benchmark for every segment comparison that follows
- Any segment above 27.9% is higher risk than average; any segment below is healthier than average
- In a real neobank context, 28% annual churn is elevated — industry benchmarks vary but 15–20% is a healthier range, which frames the business case for a retention initiative


---
## Section 4 — Churn by Plan Tier

### What this section does
Groups customers by their plan tier and computes multiple aggregations in a single `.groupby()` call: customer count, churned count, churn rate, average balance, and average products held.

### Named aggregation syntax
```python
.agg(
    customers=("churned", "count"),
    churned=("churned", "sum"),
    churn_rate=("churned", "mean"),
)
```
This is the **modern named aggregation syntax** (introduced in Pandas 0.25). The format is `output_column_name = (source_column, aggregation_function)`. It is preferred over the older `.agg({"col": "func"})` style because:
- Column names are explicit and descriptive
- Multiple different aggregations can target different source columns in one call
- The resulting DataFrame has clean, meaningful column names with no extra renaming needed

### Why `.mean()` gives churn rate
Since `churned` is a binary column (0 or 1), taking the mean gives the proportion of 1s — which is exactly the churn rate. This is a standard pandas idiom for computing rates on binary flags.


In [4]:
plan_summary = (
    df.groupby("plan_tier")
    .agg(
        customers=("churned", "count"),
        churned=("churned", "sum"),
        churn_rate=("churned", "mean"),
        avg_balance_cad=("avg_monthly_balance_cad", "mean"),
        avg_products=("num_products", "mean"),
    )
    .round({"churn_rate": 3, "avg_balance_cad": 0, "avg_products": 2})
    .sort_values("churn_rate", ascending=False)
)

plan_summary["churn_rate_pct"] = plan_summary["churn_rate"].map("{:.1%}".format)

print(plan_summary.drop(columns="churn_rate").to_string())


           customers  churned  avg_balance_cad  avg_products churn_rate_pct
plan_tier                                                                  
Free             554      226            497.0          1.51          40.8%
Standard         652      146           1920.0          1.96          22.4%
Premium          294       46           5093.0          2.98          15.6%


### Key findings
- **Free-tier: 40.8% churn** — the highest-risk segment by a wide margin. Free customers have no financial switching cost, lower product engagement, and lower balances. They are the easiest to lose.
- **Standard: 22.4% churn** — below the overall rate, but still significant volume given this is the largest tier (45% of the base)
- **Premium: 15.6% churn** — the healthiest retention, consistent with higher investment in the platform (higher fees, more products, larger balances)

### Business implication
Plan tier is the single strongest churn predictor in this dataset. A product strategy that converts Free users to Standard — even with a discounted offer — would likely have a material impact on overall churn rate.


---
## Section 5 — Churn by Engagement Signals

### What this section does
Examines two key behavioural engagement signals — **login recency** and **transaction volume** — and bins them into named segments using `pd.cut()`.

### How `pd.cut()` works
```python
df["login_recency"] = pd.cut(
    df["days_since_last_login"],
    bins=[0, 7, 21, 45, 120],
    labels=["Active (≤7d)", "Moderate (8-21d)", "Lapsing (22-45d)", "Dormant (45d+)"],
    right=True,
)
```
- `bins` defines the boundary edges of each bucket
- `labels` names each resulting bucket — these names are what appear in the output
- `right=True` (the default) means each interval includes its right edge: a customer with exactly 7 days since login falls into the "Active" bucket, not "Moderate"
- The result is a new categorical column that can be used in `groupby()` just like any other column

**Why bin continuous variables?**  
Raw continuous numbers are hard to communicate and segment. Named buckets (Active / Lapsing / Dormant) map directly to business actions and are the format used in dashboards and executive reports.


In [5]:
# ── Login recency buckets ──
df["login_recency"] = pd.cut(
    df["days_since_last_login"],
    bins=[0, 7, 21, 45, 120],
    labels=["Active (<=7d)", "Moderate (8-21d)", "Lapsing (22-45d)", "Dormant (45d+)"],
    right=True,
)

login_churn = (
    df.groupby("login_recency", observed=True)["churned"]
    .agg(customers="count", churn_rate="mean")
    .assign(churn_rate=lambda x: x["churn_rate"].map("{:.1%}".format))
)
print("Churn by Login Recency:")
print(login_churn.to_string())

# ── Transaction volume buckets ──
df["txn_segment"] = pd.cut(
    df["monthly_txn_count"],
    bins=[0, 4, 12, 25, 100],
    labels=["Very Low (<5)", "Low (5-12)", "Moderate (13-25)", "High (25+)"],
    right=True,
)

txn_churn = (
    df.groupby("txn_segment", observed=True)["churned"]
    .agg(customers="count", churn_rate="mean")
    .assign(churn_rate=lambda x: x["churn_rate"].map("{:.1%}".format))
)
print("\nChurn by Monthly Transaction Volume:")
print(txn_churn.to_string())


Churn by Login Recency:
                  customers churn_rate
login_recency                         
Active (<=7d)           527      24.9%
Moderate (8-21d)        547      28.2%
Lapsing (22-45d)        257      32.7%
Dormant (45d+)           65      40.0%

Churn by Monthly Transaction Volume:
                  customers churn_rate
txn_segment                           
Very Low (<5)            45      55.6%
Low (5-12)              550      38.7%
Moderate (13-25)        631      22.3%
High (25+)              274      14.2%


### Key findings
- **Dormant customers (45+ days since login): 40.0% churn** — the clear engagement threshold. Once a customer goes 3+ weeks without logging in, churn probability rises sharply.
- **Very Low transaction customers (<5/month): 55.6% churn** — the strongest single-feature signal in the dataset. Low transaction volume reflects that the neobank is not the customer's primary financial account.

### Business implication
Login recency and transaction frequency are both **leading indicators** — they deteriorate *before* a customer churns, giving the bank a window to intervene. A real-time trigger (e.g. push notification after 14 days of inactivity) could catch customers in the "Lapsing" bucket before they reach "Dormant".


---
## Section 6 — Churn by Balance Behaviour

### What this section does
Analyses churn rates across the three balance trend categories: growing, stable, and declining. Unlike the continuous features above, `balance_trend` is already a categorical variable, so no binning is needed.

### Why balance trend matters
A declining balance is a **behavioural signal** — it reflects the customer actively withdrawing money and reducing their relationship with the bank. By the time the balance reaches near-zero, the customer has often already mentally decided to leave. Tracking trend (not just absolute balance) gives a longer prediction window.


In [6]:
balance_churn = (
    df.groupby("balance_trend")
    .agg(
        customers=("churned", "count"),
        churn_rate=("churned", "mean"),
        avg_balance_cad=("avg_monthly_balance_cad", "mean"),
    )
    .round({"churn_rate": 3, "avg_balance_cad": 0})
    .sort_values("churn_rate", ascending=False)
)

balance_churn["churn_rate_pct"] = balance_churn["churn_rate"].map("{:.1%}".format)
print("Churn by Balance Trend:")
print(balance_churn.drop(columns="churn_rate").to_string())


Churn by Balance Trend:
               customers  avg_balance_cad churn_rate_pct
balance_trend                                           
declining            390           2140.0          35.6%
stable               648           2014.0          27.2%
growing              462           1916.0          22.3%


### Key findings
- **Declining balance: 35.6% churn** — 13 percentage points above the overall rate
- **Growing balance: 22.3% churn** — below the overall rate; customers actively building their balance are invested in the relationship
- Note that average balance is similar across all three groups (~$2,000), confirming that the *direction* of change matters more than the *absolute level*

### Business implication
Balance trend monitoring should be a core feature of a churn early-warning system. A customer whose balance has declined three months in a row is a priority for outreach — even if their current balance looks fine.


---
## Section 7 — Tenure Cohort Analysis

### What this section does
Segments customers into four tenure buckets and compares churn rates across them. This is a simplified form of **cohort analysis** — a standard technique in subscription and fintech businesses.

### What cohort analysis reveals
Cohort analysis answers the question: *does churn risk change as a customer gets older?* In most subscription products, the answer is yes — newer customers churn more because:
1. They haven't yet integrated the product into their daily habits
2. They may have signed up speculatively (e.g. for a promotion)
3. They haven't received enough value to feel "locked in"

This has a direct implication for where to invest in the customer journey.


In [7]:
df["tenure_bucket"] = pd.cut(
    df["tenure_months"],
    bins=[0, 6, 18, 36, 72],
    labels=["<6 months", "6-18 months", "18-36 months", "36+ months"],
)

tenure_churn = (
    df.groupby("tenure_bucket", observed=True)
    .agg(
        customers=("churned", "count"),
        churn_rate=("churned", "mean"),
        avg_products=("num_products", "mean"),
        avg_balance_cad=("avg_monthly_balance_cad", "mean"),
    )
    .round({"churn_rate": 3, "avg_products": 2, "avg_balance_cad": 0})
)

tenure_churn["churn_rate_pct"] = tenure_churn["churn_rate"].map("{:.1%}".format)
print("Churn by Tenure Cohort:")
print(tenure_churn.drop(columns="churn_rate").to_string())


Churn by Tenure Cohort:
               customers  avg_products  avg_balance_cad churn_rate_pct
tenure_bucket                                                         
<6 months            220          1.95           1825.0          33.6%
6-18 months          402          2.04           1969.0          29.4%
18-36 months         489          2.02           2147.0          25.6%
36+ months           389          1.94           2009.0          26.0%


### Key findings
- **<6 months: 33.6% churn** — the highest-risk cohort, 6 points above the overall rate
- **36+ months: 26.0% churn** — loyal customers still churn, but at a lower rate
- `avg_products` is relatively consistent across cohorts (~2.0), meaning product depth alone doesn't explain the tenure effect — it is genuinely about relationship maturity

### Business implication
The first 6 months are the most critical retention window. This points to onboarding as the highest-ROI area to invest: a structured onboarding sequence that drives product adoption (e.g. getting a new user to link their external bank and set up a savings goal within 30 days) would likely reduce early-tenure churn significantly.


---
## Section 8 — High-Risk Segment Identification

### What this section does
Uses a **boolean mask** to filter the dataset to customers who simultaneously match multiple churn risk factors. This identifies a priority intervention segment — the customers most likely to churn if not acted on.

### How boolean masking works
```python
at_risk = df[
    (df["plan_tier"] == "Free")
    & (df["balance_trend"] == "declining")
    & (df["days_since_last_login"] > 21)
    & (df["num_products"] == 1)
]
```
Each condition in parentheses returns a **boolean Series** (a column of True/False values). The `&` operator combines them element-wise — a row is included in the result only if *all* conditions are True for that row.

**Why parentheses around each condition?**  
Python's operator precedence treats `&` as higher priority than `==`, `>`, etc. Without parentheses, `df["plan_tier"] == "Free" & df["balance_trend"] == "declining"` would be evaluated incorrectly. Always wrap each condition.

### Criteria selection rationale
Each criterion was chosen because it independently showed elevated churn in previous sections:
- `Free` plan — 40.8% churn (Section 4)
- `declining` balance — 35.6% churn (Section 6)
- `> 21 days` since login — crosses into "Lapsing" territory (Section 5)
- `num_products == 1` — no cross-sell stickiness (no additional products tying them to the platform)


In [8]:
at_risk = df[
    (df["plan_tier"] == "Free")
    & (df["balance_trend"] == "declining")
    & (df["days_since_last_login"] > 21)
    & (df["num_products"] == 1)
].copy()

print("HIGH-RISK SEGMENT PROFILE")
print("=" * 40)
print(f"Segment size    : {len(at_risk):,} customers ({len(at_risk)/total:.1%} of base)")
print(f"Churn rate      : {at_risk['churned'].mean():.1%}")
print(f"Avg balance     : ${at_risk['avg_monthly_balance_cad'].mean():,.0f} CAD")
print(f"Avg tenure      : {at_risk['tenure_months'].mean():.0f} months")
print(f"Avg days inactive: {at_risk['days_since_last_login'].mean():.0f} days")
print(f"Avg txn/month   : {at_risk['monthly_txn_count'].mean():.1f}")


HIGH-RISK SEGMENT PROFILE
Segment size    : 14 customers (0.9% of base)
Churn rate      : 78.6%
Avg balance     : $574 CAD
Avg tenure      : 29 months
Avg days inactive: 39 days
Avg txn/month   : 8.1


### Key findings
- This segment's **78.6% churn rate** is nearly 3× the overall rate of 27.9%
- Despite having average tenure of ~29 months, these customers are deeply disengaged — they are long-tenured but inactive, which in neobank terms often means the account has become a "zombie" account (open but unused)

### Business implication
This is the list you hand to a CRM or growth team for a targeted re-engagement campaign. A campaign could offer a Free → Standard upgrade incentive (e.g. 2 months free), a personalised in-app message, or a product education push. The segment is small enough to treat individually and high-risk enough that intervention has clear ROI.


---
## Section 9 — Revenue Impact Estimate

### What this section does
Translates churn from a percentage into a dollar figure by estimating **MRR lost to churn** and a simplified **customer lifetime value (LTV)**. This is the step that makes the analysis boardroom-ready — executives and product managers respond to revenue figures, not churn percentages.

### The LTV formula used
```python
estimated_ltv = avg_fee_paid / churn_rate
```
This is the **simple LTV formula for subscription businesses**: Average Monthly Revenue ÷ Monthly Churn Rate. It assumes:
- Constant monthly revenue per customer
- Constant churn rate over time
- No time-value-of-money discounting

It is approximate, but directionally correct and sufficient for a portfolio analysis. More sophisticated LTV models (e.g. using survival analysis or discounted cash flows) would be used in a production context.

### The "5 percentage point reduction" framing
```python
int(total * 0.05)  # customers saved by a 5pp churn reduction
avg_fee_paid * int(total * 0.05)  # MRR recovered
```
Rather than saying "churn is 27.9%", framing the output as "a 5pp reduction saves X customers and recovers $Y/month in MRR" gives stakeholders a concrete target and a business case for investment. This is standard practice in retention analytics.


In [9]:
churned_df = df[df["churned"] == 1]

mrr_lost = churned_df["monthly_fee_cad"].sum()
mrr_retained = df[df["churned"] == 0]["monthly_fee_cad"].sum()

avg_fee_paid = df[df["plan_tier"] != "Free"]["monthly_fee_cad"].mean()
estimated_ltv = avg_fee_paid / churn_rate

customers_saved_5pp = int(total * 0.05)
mrr_recovered = avg_fee_paid * customers_saved_5pp

print("REVENUE IMPACT")
print("=" * 45)
print(f"  MRR lost to churn (paid plans) : ${mrr_lost:>10,.2f} CAD/month")
print(f"  MRR retained                   : ${mrr_retained:>10,.2f} CAD/month")
print(f"  Estimated avg LTV (paid)        : ${estimated_ltv:>10,.2f} CAD")
print()
print(f"  A 5pp churn reduction would save ~{customers_saved_5pp} customers")
print(f"  and recover ~${mrr_recovered:,.0f} CAD/month in MRR")


REVENUE IMPACT
  MRR lost to churn (paid plans) : $  2,378.08 CAD/month
  MRR retained                   : $ 10,012.46 CAD/month
  Estimated avg LTV (paid)        : $     47.00 CAD

  A 5pp churn reduction would save ~75 customers
  and recover ~$982 CAD/month in MRR


### Key findings
- **~$2,378 CAD/month in MRR** is lost to churn on paid plans — this is recurring revenue that could be retained with effective intervention
- **Estimated LTV ~$47 CAD** per paid customer is the upper bound on what it is worth spending per customer to retain them (i.e. a retention campaign should cost less than $47 per saved customer to be ROI-positive)
- Reducing churn by 5 percentage points recovers ~$982 CAD/month — a concrete goal for a retention initiative

### Note on LTV simplification
This LTV estimate is conservative. A full model would account for:
- Revenue from non-fee sources (interchange, FX margins, investment products)
- Referral value — retained customers who refer others
- Product upsell over time (Free → Standard → Premium)
In practice, true LTV for a neobank customer is likely 3–5× this estimate.


---
## Summary of Findings

| Finding | Metric | Business Implication |
|---|---|---|
| Free-tier churn rate | 40.8% | Freemium conversion is the #1 retention lever |
| Dormant users (45d+) churn rate | 40.0% | Re-engagement trigger needed at 14–21 days inactivity |
| Very low transaction users churn rate | 55.6% | Low transaction volume = neobank is not primary account |
| Declining balance customers churn rate | 35.6% | Balance trend monitoring = early warning signal |
| New customers (<6 months) churn rate | 33.6% | Onboarding experience is the critical investment window |
| High-risk segment churn rate | 78.6% | Priority CRM intervention list identified |
| MRR lost to churn monthly | ~$2,378 CAD | Revenue case for retention investment |

---

## Recommended Next Steps

1. **Onboarding redesign** — reduce <6 month churn by creating a structured 30-day activation sequence that drives customers to link an external bank and set up a second product (savings goal or crypto wallet)
2. **Inactivity trigger** — deploy a push notification / email sequence at 14 days of inactivity, before customers reach the "Dormant" threshold
3. **Free → Standard conversion campaign** — target the high-risk segment (Free + declining + inactive + single product) with a discounted upgrade offer; at $47 LTV, even a $15 incentive is ROI-positive if it converts
4. **Balance trend alerting** — build a real-time flag for customers with 2+ consecutive months of declining balance; route to proactive outreach queue
5. **Expand this analysis** — next step is a logistic regression model to assign each customer an individual churn probability score for production use in a CRM

---
*Dataset: synthetic, generated with Python/NumPy for portfolio purposes. All figures are illustrative.*
